# Adaptive Network Traffic Anomaly Classification

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,roc_auc_score,confusion_matrix

In [ ]:
df=pd.read_csv('../data/network_flows.csv'); print(df.head()); print(df.isna().sum()); print('duplicates:',df.duplicated().sum()); print(df['anomaly'].value_counts(normalize=True))

In [ ]:
X=df.drop(columns='anomaly'); y=df['anomaly']; num=[c for c in X.columns if c!='protocol']; cat=['protocol']
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
pipe=Pipeline([('pre',pre),('clf',LogisticRegression(max_iter=1200,class_weight='balanced'))]); Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y); pipe.fit(Xtr,ytr); p=pipe.predict(Xte); print(classification_report(yte,p)); print('ROC-AUC',roc_auc_score(yte,pipe.predict_proba(Xte)[:,1])); print(confusion_matrix(yte,p))